Connect to drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ls /content/drive/MyDrive

AgriConceptNER@  MyAgriNER_copy/


In [ ]:
rm -rf /content/drive/MyDrive/MyAgriNER_copy

Continue

In [ ]:
%env PJ_DIR=/content/drive/My Drive/MyAgriNER_copy
!echo ${PJ_DIR}/data

env: PJ_DIR=/content/drive/My Drive/MyAgriNER_copy
/content/drive/My Drive/MyAgriNER_copy/data


Git clone the NCRFPP git repo

In [ ]:
cd /

In [ ]:
import os
if not os.path.exists('NCRFpp'):
    !git clone https://github.com/jiesutd/NCRFpp.git
else:
    print('Repository already exists.')

Cloning into 'NCRFpp'...
remote: Enumerating objects: 768, done.
remote: Total 768 (delta 0), reused 0 (delta 0), pack-reused 768 (from 1)
Receiving objects: 100% (768/768), 6.89 MiB | 19.07 MiB/s, done.
Resolving deltas: 100% (484/484), done.


In [ ]:
!ls

drive  NCRFpp  sample_data


In [ ]:
!cp -r NCRFpp/* -u "${PJ_DIR}"

In [ ]:
import shutil
import os

# Source folder path
folder_path = '/content/drive/MyDrive/AgriConceptNER'

# TODO: Specify your desired destination path here
destination_path = '/content/drive/MyDrive/MyAgriNER_copy/' # Example: '/content/drive/My Drive/MyAgriNER_copy/'

# Create the destination directory if it doesn't exist
os.makedirs(destination_path, exist_ok=True)

# Copy the folder to the destination
# If the destination directory already exists, copytree will merge content, or raise an error if not empty based on content.
# For a fresh copy, ensure destination_path points to where you want the 'AgriConceptNER' folder itself to reside.
# Example: if folder_path is '/A/B' and destination_path is '/C', it will create '/C/B'

# To copy the content of AgriConceptNER directly into destination_path, use:
for item in os.listdir(folder_path):
    s = os.path.join(folder_path, item)
    d = os.path.join(destination_path, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)


print(f"Folder '{folder_path}' copied to '{destination_path}'")


Folder '/content/drive/MyDrive/AgriConceptNER' copied to '/content/drive/MyDrive/MyAgriNER_copy/'


In [ ]:
!rm -rf "${PJ_DIR}"

Add requirements.py and install the required libraries

In [ ]:
with open('requirements.txt', 'w') as f:
    f.write('torch\nnumpy\n')
!pip install -r requirements.txt

Unzip the data file into project directory

In [ ]:
!unzip -o "/content/drive/My Drive/MyAgriNER/data.zip" -d /NCRFpp

Archive:  /content/drive/My Drive/MyAgriNER/data.zip
   creating: /NCRFpp/data/
  inflating: /NCRFpp/data/first_sem_agri_bio_syllable.conll  
  inflating: /NCRFpp/data/first_sem_agri_bio_word.conll  
  inflating: /NCRFpp/data/first_sem_agri_bioes_syllable.conll  
  inflating: /NCRFpp/data/first_sem_agri_bioes_word.conll  
  inflating: /NCRFpp/data/burmese_agri_syllable.emb  
  inflating: /NCRFpp/data/burmese_agri_word.emb  


check if the data files are there

In [ ]:
!ls "${PJ_DIR}/model"

charbigru.py   charcnn.py  __init__.py	      seqlabel.py  wordsequence.py
charbilstm.py  crf.py	   sentclassifier.py  wordrep.py


Slplit files into folder with train, dev and test

In [ ]:
import os
import random
import glob

def split_conll_file(file_path, train_ratio=0.8, dev_ratio=0.1):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    # CoNLL files usually separate sentences/sequences with empty lines
    sequences = content.split('\n\n')
    random.seed(42)  # For reproducibility
    random.shuffle(sequences)

    total = len(sequences)
    train_end = int(total * train_ratio)
    dev_end = train_end + int(total * dev_ratio)

    train_data = sequences[:train_end]
    dev_data = sequences[train_end:dev_end]
    test_data = sequences[dev_end:]

    # Create a subfolder based on the filename
    file_name = os.path.basename(file_path)
    folder_name = file_name.replace('.conll', '')
    target_dir = os.path.join(os.path.dirname(file_path), folder_name)
    os.makedirs(target_dir, exist_ok=True)

    splits = {
        f'train.{file_name}': train_data,
        f'dev.{file_name}': dev_data,
        f'test.{file_name}': test_data
    }

    for out_name, data in splits.items():
        out_path = os.path.join(target_dir, out_name)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(data) + '\n')
        print(f"Saved {len(data)} sequences to {out_path}")

# Dynamically process all .conll files in the directory that are not already splits
data_dir = '/NCRFpp/data'
conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
               if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for file in conll_files:
    split_conll_file(file)

Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllab

In [ ]:
# rm /NCRFpp/*.config

In [ ]:
import os
import glob

# Get project root from environment variable, default to /NCRFpp if not set
project_root = os.environ.get('PJ_DIR', '/NCRFpp')

def create_config(name, train_path, dev_path, test_path):
    config_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir={project_root}/{train_path}
dev_dir={project_root}/{dev_path}
test_dir={project_root}/{test_path}
model_dir={project_root}/models/{name}
# word_emb_dir=data/sample.word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=200
char_emb_dim=200

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=ADAM
iteration=30
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.3
lstm_layer=1
bilstm=True
learning_rate=0.001
lr_decay=0
momentum=0
l2=1e-8
gpu
clip=5.0
"""
    # Save config in the project root directory
    config_path = os.path.join(project_root, f"{name}.train.config")
    with open(config_path, 'w') as f:
        f.write(config_content.strip())
    print(f"Created config: {config_path}")

# Dynamic detection of files in data directory
data_dir = os.path.join(project_root, 'data')
os.makedirs(os.path.join(project_root, 'models'), exist_ok=True)

# Find all original conll names (e.g., bio.conll, bio_syl.conll)
conll_files = [f for f in os.listdir(data_dir) if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for base_file in conll_files:
    name = base_file.replace('.conll', '')

    # Construct paths relative to the execution context (NCRFpp folder)
    # Now pointing to the subfolders created by the splitter
    train_p = f"data/{name}/train.{base_file}"
    dev_p = f"data/{name}/dev.{base_file}"
    test_p = f"data/{name}/test.{base_file}"

    # Verify files exist before creating config
    if all(os.path.exists(os.path.join(project_root, p)) for p in [train_p, dev_p, test_p]):
        create_config(name, train_p, dev_p, test_p)
    else:
        print(f"Skipping {name}: Missing split files in subfolder {name}.")

Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bio_syllable.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bioes_word.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bio_word.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bioes_syllable.train.config


In [ ]:
import os
import glob

# Get project root from environment variable, default to /NCRFpp if not set
project_root = os.environ.get('PJ_DIR', '/NCRFpp')

def create_config(name, train_path, dev_path, test_path):
    word_emb_dir_setting = ""
    if "word" in name:
        word_emb_dir_setting = f"word_emb_dir={project_root}/data/burmese_agri_word.emb"
    elif "syllable" in name:
        word_emb_dir_setting = f"word_emb_dir={project_root}/data/burmese_agri_syllable.emb"

    config_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir={project_root}/{train_path}
dev_dir={project_root}/{dev_path}
test_dir={project_root}/{test_path}
model_dir={project_root}/models/with_emb.{name}
{word_emb_dir_setting}

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=200
char_emb_dim=200

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=ADAM
iteration=30
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.3
lstm_layer=1
bilstm=True
learning_rate=0.001
lr_decay=0
momentum=0
l2=1e-8
gpu
clip=5.0
"""
    # Save config in the root NCRFpp directory
    config_path = os.path.join(project_root, f"{name}.with_emb.train.config")
    with open(config_path, 'w') as f:
        f.write(config_content.strip())
    print(f"Created config: {config_path}")

# Dynamic detection of files in data directory
data_dir = os.path.join(project_root, 'data')
os.makedirs(os.path.join(project_root, 'models'), exist_ok=True)

# Find all original conll names (e.g., bio.conll, bio_syl.conll)
conll_files = [f for f in os.listdir(data_dir) if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for base_file in conll_files:
    name = base_file.replace('.conll', '')

    # Construct paths relative to the execution context (NCRFpp folder)
    # Now pointing to the subfolders created by the splitter
    train_p = f"data/{name}/train.{base_file}"
    dev_p = f"data/{name}/dev.{base_file}"
    test_p = f"data/{name}/test.{base_file}"

    # Verify files exist before creating config
    if all(os.path.exists(os.path.join(project_root, p)) for p in [train_p, dev_p, test_p]):
        create_config(name, train_p, dev_p, test_p)
    else:
        print(f"Skipping {name}: Missing split files in subfolder {name}.")

Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bio_syllable.with_emb.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bioes_word.with_emb.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bio_word.with_emb.train.config
Created config: /content/drive/My Drive/MyAgriNER_copy/first_sem_agri_bioes_syllable.with_emb.train.config


In [ ]:
!cat "${PJ_DIR}/first_sem_agri_bioes_syllable.train.config"

### use # to comment out the configure item

### I/O ###
train_dir=/content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
dev_dir=/content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
test_dir=/content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
model_dir=/content/drive/My Drive/MyAgriNER_copy/models/first_sem_agri_bioes_syllable
# word_emb_dir=data/sample.word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=200
char_emb_dim=200

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=ADAM
iteration=30
batch_size=10
ave_batch_loss=False

###Hyperparame

In [ ]:
!head -n 200 "/content/drive/My Drive/MyAgriNER/NCRFpp_backup/20260611_095916/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll"

ကာဘင်	B-FUNG
ဒါဇင်	E-FUNG
,	O
မန်	B-FUNG
ကို	I-FUNG
ဇက်	E-FUNG
,	O
ကလို	B-FUNG
ရိုသာလိုနေး	E-FUNG
,	O
သိုင်အိုဖာနိတ်မီ	B-FUNG
သိုင်း	E-FUNG
ဆေး	O
တစ်	O
မျိုးမျိူးဖျန်း	O
ပေး	O
ပြီး	O
ရောဂါရပ်တန့်	O
သွား	O
မ	O
သွား	O
ဆက်လက်စောင့်	O
ကြည့်	O
လိူ့	O
လည်း	O
ရ	O
ပါ	O
တယ်	O
။	O

ကြဲပက်	S-FARM_OP
စိုက်ပျိုး	B-FARM_OP
ခြင်း	E-FARM_OP
-	O
(	O
၁၀	S-COUNT
-	O
၁၅	S-COUNT
)ထုပ်	O
(	O
1ထုပ်	S-QTY
)	O
။	O

ကော့ပါး	S-FUNG
ဆေးဖျန်း	O
ပါ	O
။	O

တစ်	B-PERIOD
နှစ်	E-PERIOD
ကျော်ကြာ	O
နေ	O
တဲ့	O
နွား	B-FERT
သေး	E-FERT
ဗူး	O
ထဲ	O
ထည့်	O
ထား	O
တာ	O
နွား	O
မ	O
ရှိ	O
တော့	O
လို့	O
ခုချိန်	O
သုံး	O
ရင်	O
ရ	O
နိုင်	O
သေးလား	O
ခင်ဗျာ	O
။	O

အပင်	O
ပျော့	O
ဖြစ်	O
ပြီး	O
ပင်	B-CROP_PART
စည်	E-CROP_PART
အောက်ခြေ	O
တွင်	O
ဥအု	O
သည်	O
။	O

ဓား	S-EQUIP
ခွဲရာတွင်	O
မှို	S-CROP
စိုက်	S-FARM_OP
ထုပ်	O
၏	O
အပေါ်ယံပလပ်စတစ်သား	O
ပြဲ	O
ရုံ	O
သာ	O
ခွဲ	O
ရ	O
မည်	O
။	O

သခွား	S-CROP
မျိုး	O
များ	O
သည်	O
မြေချဉ်ငံ	O
ဓါတ်(၅.၅)	O
မှ	O
(၆.၅)	O
အတွင်း	O
ပိုမို	O
ဖြစ်ထွန်း	O
ပါ	O
သည်	O
။	O

ပဲ	B-CROP
ပင်	E-CROP
အရွှက်	O
မှာ	O
ပုံ	O
ထဲ	

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_word.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_word.emb
Embedding:
     pretrain word:60670, prefect match:10468, case_match:0, oov:4450, oov%:0.29827736443461356
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: 

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bioes_word.train.config"

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /content/drive/My Drive/MyAgriNER/NCRFpp_backup/20260611_095916/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /content/drive/My Drive/MyAgriNER/NCRFpp_backup/20260611_095916/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /conten

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_syllable.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_syllable.emb
Embedding:
     pretrain word:17187, prefect match:2217, case_match:0, oov:237, oov%:0.09653767820773931
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    fil

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bioes_syllable.train.config"

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /content/drive/My Drive/MyAgriNER_copy/dat

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bio_word.with_emb.train.config"

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /content/drive/My Drive/MyAgriNER_copy/data/burmese_agri_word.emb
Embedding:
     pretrain word:60670, prefect match:10468, case_match:0, oov:4450, oov%:0.29827736443461356
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /content/drive/My Drive/MyAgriNER_copy/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /content/drive/My Drive/MyAgriNER_copy/data/f

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bio_word.train.config"

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bio_word

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bio_syllable.with_emb.train.config"

In [ ]:
!cd "${PJ_DIR}" && python "./main.py" --config "./first_sem_agri_bio_syllable.train.config"

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
     Test   file directory: /content/drive/My Drive/MyAgriNER_copy/data/first_se

### Decoding Configuration Generator
This script generates `.decode.config` files for testing. It assumes that training has completed and produced model files in the `/NCRFpp/models/` directory.

In [ ]:
import os
import glob

def create_decode_config(name, test_path, model_path, dset_path):
    # Ensure output directory exists
    os.makedirs('/NCRFpp/output', exist_ok=True)

    decode_content = f"""
### I/O ###
status=decode
raw_dir=/NCRFpp/{test_path}
decode_dir=/NCRFpp/output/{name}.test.out
dset_dir={dset_path}
load_model_dir={model_path}

nbest=10
#gpu
"""
    config_path = f"/NCRFpp/{name}.decode.config"
    with open(config_path, 'w') as f:
        f.write(decode_content.strip())
    print(f"Created decode config: {config_path}")

# Updated detection logic for models in the root models folder
models_root = '/NCRFpp/models'
if os.path.exists(models_root):
    # Find all .dset files to identify trained datasets
    dset_paths = glob.glob(os.path.join(models_root, "*.dset"))

    for dset_path in dset_paths:
        # e.g., /NCRFpp/models/bio.dset -> name = 'bio'
        name = os.path.basename(dset_path).replace('.dset', '')

        # Find matching models for this name (e.g., bio.0.model, bio.1.model)
        model_files = sorted(glob.glob(os.path.join(models_root, f"{name}.*.model")))

        if model_files:
            # Use the latest model and the standard test path
            test_p = f"data/{name.replace('with_emb.', '')}/test.{name.replace('with_emb.', '')}.conll"
            create_decode_config(name, test_p, model_files[-1], dset_path)
        else:
            print(f"No models found for {name} in {models_root}")
else:
    print(f"Models directory {models_root} does not exist.")

Created decode config: /NCRFpp/with_emb.first_sem_agri_bioes_word.decode.config


In [ ]:
ls /NCRFpp

data/
first_sem_agri_bioes_syllable.train.config
first_sem_agri_bioes_word.decode.config
first_sem_agri_bioes_word.train.config
first_sem_agri_bio_syllable.train.config
first_sem_agri_bio_word.train.config
models/
output/


In [ ]:
# rm /NCRFpp/demo.with_emb.first_sem_agri_bioes_word.decode.config

In [ ]:
ls /NCRFpp/models

ls: cannot access '/NCRFpp/models': No such file or directory


In [ ]:
ls /NCRFpp/output

first_sem_agri_bioes_word.test.out  with_emb.first_sem_agri_bioes_word.test.out


In [ ]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bioes_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 200
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bi

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bio

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bioes_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /NCR

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_sylla

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bio_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_word/test.fir

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
    

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bio_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
     Test   file directory: /NCRFpp/data/first

In [ ]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_syllable.decode.config

python3: can't open file '/content/NCRFpp/main.py': [Errno 2] No such file or directory


In [ ]:
ls /NCRFpp/output/

first_sem_agri_bioes_syllable.test.out
first_sem_agri_bioes_word.test.out
first_sem_agri_bio_syllable.test.out
first_sem_agri_bio_word.test.out
with_emb.first_sem_agri_bioes_syllable.test.out
with_emb.first_sem_agri_bioes_word.test.out
with_emb.first_sem_agri_bio_syllable.test.out
with_emb.first_sem_agri_bio_word.test.out


In [ ]:
!head /NCRFpp/output/first_sem_agri_bioes_word.test.out -n 200

# 0.5436 0.1349 0.0715 0.0691 0.0576 0.0342 0.0262 0.0243 0.0195 0.0191
သို့ O S-CROP S-FARM_OP B-CROP S-CROP_PART B-FARM_OP S-PEST S-WEATHER B-FERT O
ဖြစ် O O O E-CROP O E-FARM_OP O O E-FERT S-FARM_OP
၍ O O O O O O O O O O
စပါး O O O O O O O O O O
ပင် O O O O O O O O O O
များ O O O O O O O O O O
၏ O O O O O O O O O O
အမြစ် O O O O O O O O O O
ဇုန်ဝန်းကျင် O O O O O O O O O O
သို့ O O O O O O O O O O
ရေ O O O O O O O O O O
ကို O O O O O O O O O O
စိုစွတ် O O O O O O O O O O
ရုံ O O O O O O O O O O
သာ O O O O O O O O O O
ပေးသွင်းရုံ O O O O O O O O O O
ဖြင့် O O O O O O O O O O
စိုက်ပျိုး O O O O O O O O O O
အောင်မြင် O O O O O O O O O O
နိုင် O O O O O O O O O O
ခြင်း O O O O O O O O O O
ဖြစ် O O O O O O O O O O
သည် O O O O O O O O O O
။ O O O O O O O O O O

# 0.5412 0.1343 0.0712 0.0688 0.0573 0.0340 0.0260 0.0242 0.0234 0.0195
မြေပြုပြင် O S-CROP S-FARM_OP B-CROP S-CROP_PART B-FARM_OP S-PEST S-WEATHER O B-FERT
ခြင်း O O O E-CROP O E-FARM_OP O O O E-FERT
အမြစ် O O O O O O O O O O
ကောင

In [ ]:
import shutil
import os
from datetime import datetime

# Define the destination path in Google Drive
drive_export_path = '/content/drive/My Drive/MyAgriNER/'

# Create the main destination folder for the NCRFpp backup
ncrfpp_backup_path = os.path.join(drive_export_path, 'NCRFpp_backup')
os.makedirs(ncrfpp_backup_path, exist_ok=True)

# Source directory to backup (the entire /NCRFpp)
src_ncrfpp_dir = '/NCRFpp'
dst_ncrfpp_dir = os.path.join(ncrfpp_backup_path, datetime.now().strftime('%Y%m%d_%H%M%S'))

print(f"Starting backup of {src_ncrfpp_dir} to {dst_ncrfpp_dir}...")

if os.path.exists(src_ncrfpp_dir):
    # If the destination for this specific timestamped backup already exists (unlikely, but for robustness)
    if os.path.exists(dst_ncrfpp_dir):
        shutil.rmtree(dst_ncrfpp_dir)

    shutil.copytree(src_ncrfpp_dir, dst_ncrfpp_dir)
    print(f"\nBackup complete! The entire {src_ncrfpp_dir} directory is now safe in {dst_ncrfpp_dir}.")
else:
    print(f"Error: Source directory {src_ncrfpp_dir} does not exist.")

Starting backup of /NCRFpp to /content/drive/My Drive/MyAgriNER/NCRFpp_backup/20260611_095916...

Backup complete! The entire /NCRFpp directory is now safe in /content/drive/My Drive/MyAgriNER/NCRFpp_backup/20260611_095916.
